# Intro 

The main objectives of this report is to see how the tensor products behaves throught the simulations and how it preforms acoutdning diferent wheights matrix 

In [ ]:
import os 
os.chdir("..")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import Ridge
import statsmodels.api as sm
from itertools import product
import pandas as pd
from patsy import dmatrix
from st_repl import SpatialReg
import patsy
import geopandas as gpd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
from shapely.geometry import Polygon


sr = SpatialReg()


In [ ]:
gdf = sr.spatial_panel(mu=2, sigma=3,time=100,rho=0.7, seed=787)
gdf

In [ ]:
df = gdf[gdf["time"] == 0]
# -----------------------------------

# 2. Configure academic style settings
# Use a serif font (like Times New Roman or Computer Modern) to match LaTeX
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    }
)

# 3. Create the figure and axis with a publication-ready size (e.g., width for a single column)
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

# 4. Plot the GeoDataFrame using the 'viridis' colormap
df.plot(
    column="zipcode",
    ax=ax,
    cmap="viridis",  # Updated from "Blues" to "viridis"
    edgecolor="#333333",
    linewidth=0.6,
    alpha=0.85,
    legend=False,
)

# 5. Add ZIP code labels at the centroid of each polygon
for idx, row in df.iterrows():
    # Find the representative point (ensures label stays inside complex polygons)
    centroid = row["geometry"].representative_point()

    # Optional: adjust label appearance (halo effect helps readability)
    ax.text(
        centroid.x,
        centroid.y,
        str(row["zipcode"][2:]),
        fontsize=8,
        ha="center",
        va="center",
        color="#111111",
        weight="bold",
        bbox=dict(
            boxstyle="round,pad=0.2",
            facecolor="white",
            alpha=0.7,
            edgecolor="none",
        ),
    )

# 6. Professional formatting (Axes, Titles, Grid)
ax.set_title(
    "Spatial Distribution of Study Areas by ZIP Code", pad=12, weight="bold"
)
ax.set_xlabel("Longitude (Degrees)", labelpad=8)
ax.set_ylabel("Latitude (Degrees)", labelpad=8)

# Clean grid lines for spatial reference
ax.grid(True, linestyle="--", alpha=0.5, color="#cccccc")
ax.set_axisbelow(True)

# Remove unnecessary top/right spines for a cleaner academic look
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# 7. Layout optimization and export
plt.tight_layout()

# Save as PDF (Vector graphic is essential for LaTeX/thesis quality)
#plt.savefig("docs/reports/01zipcode_grid_map.png", format="png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# Tensor model 

formula= "y_true ~ X_1 + X_2 + X_3 + te(cr(lat, df=6), cr(lon, df=6), constraints='center')"

# 1. Generate design matrices
y, design = patsy.dmatrices(formula, gdf)

# 2. Fit the Ridge model
model = Ridge(alpha=0.01)
model.fit(design, y)

# 3. Predict the trend/spatial surface
gdf['y_tensor'] = model.predict(design)


In [ ]:
for m in ["w_rook", "w_queen", "w_knn6"]:
    xb = gdf[["X_1","X_2","X_3",m]].values.reshape(-1,4)
    y_d = gdf["y_true"].values.reshape(-1,1)
    X = sm.add_constant(xb)
    results = sm.OLS(y_d, X).fit()
    gdf[f"y_{m.split("_")[1]}"] = results.predict(X)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score

# 1. Publication styling configuration
plt.rcParams.update({
    "font.family": "serif",          # Standard academic serif font (or 'sans-serif')
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 300,               # High resolution for print/PDF
})

sns.set_theme(style="ticks")

# 2. Define models, display names, and a professional color palette
models = [
    ("y_rook", "Spatial Lag (Rook)", "#4c72b0"),     # Muted Blue
    ("y_queen", "Spatial Lag (Queen)", "#dd8452"),  # Muted Orange
    ("y_knn6", "Spatial Lag (KNN-6)", "#55a868"),   # Muted Green
    ("y_tensor", "Tensor Product Spline", "#c44e52") # Muted Red
]

# 3. Initialize figure layout (2x2 grid)
fig, axes = plt.subplots(2, 2, figsize=(8, 8), sharex=True, sharey=True)
axes = axes.flatten()

y_true = gdf["y_true"].values

# Determine uniform axis boundaries across all subplots for fair visual comparison
all_vals = np.concatenate([y_true] + [gdf[col].values for col, _, _ in models])
lims = [np.min(all_vals), np.max(all_vals)]
# Add a 2% padding margin
padding = (lims[1] - lims[0]) * 0.02
lims = [lims[0] - padding, lims[1] + padding]

for i, (col, title, color) in enumerate(models):
    ax = axes[i]
    y_pred = gdf[col].values
    
    # Compute metrics for the text box
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Scatter plot with high-academic legibility styling
    ax.scatter(
        y_true, y_pred, 
        color=color, 
        alpha=0.5, 
        edgecolor="none", 
        s=16, 
        rasterized=True # Keeps vector PDFs lightweight
    )
    
    # 45-degree reference parity line (y = x)
    ax.plot(lims, lims, linestyle="--", color="black", linewidth=1, alpha=0.6, zorder=3)
    
    # Formatting axes and boundaries
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect("equal", adjustable="box")
    
    # Titles and labels
    ax.set_title(title, fontweight="bold", pad=8)
    if i >= 2:
        ax.set_xlabel("Observed ($y_{true}$)")
    if i % 2 == 0:
        ax.set_ylabel("Predicted ($y$)" )
        
    # Embed performance metrics inside the plot frame
    text_str = f"$R^2$ = {r2:.3f}\nRMSE = {rmse:.3f}"
    props = dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#cccccc", alpha=0.85)
    ax.text(
        0.05, 0.95, text_str, 
        transform=ax.transAxes, 
        fontsize=9, 
        verticalalignment="top", 
        bbox=props
    )
    
    # Clean spines
    sns.despine(ax=ax, top=False, right=False)
    ax.grid(True, linestyle=":", alpha=0.5, color="#aaaaaa")

# Layout optimization
fig.tight_layout()

# Save explicitly for thesis inclusion (vector PDF format is preferred for LaTeX/Word)
plt.savefig("model_comparison_grid.pdf", bbox_inches="tight")
plt.savefig("model_comparison_grid.png", bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from scipy.interpolate import griddata

# 1. Advanced Academic Styling Configuration (Removed invalid ztick key)
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 300,
    "text.usetex": False,  # Set to True if your system has a LaTeX distribution installed
})

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

# Extract coordinates and tensor predictions (Assumes `gdf` is defined in your environment)
x = gdf["lon"].values
y = gdf["lat"].values
z = gdf["y_tensor"].values

# 2. High-Resolution Grid Interpolation for Smooth Surfaces
xi = np.linspace(x.min(), x.max(), 150)
yi = np.linspace(y.min(), y.max(), 150)
X, Y = np.meshgrid(xi, yi)
Z = griddata((x, y), z, (X, Y), method='cubic')

# 3. Add Light Source for 3D Relief and Depth Perception
ls = LightSource(azdeg=315, altdeg=45)
rgb = ls.shade(Z, cmap=plt.cm.viridis, vert_exag=0.1, blend_mode='soft')

# Plot the 3D surface with lighting effects
surf = ax.plot_surface(
    X, Y, Z, 
    facecolors=rgb,         
    linewidth=0, 
    antialiased=True, 
    shade=False             
)

# 4. Colorbar Refinement
m = plt.cm.ScalarMappable(cmap="viridis")
m.set_array(Z)
cbar = fig.colorbar(m, ax=ax, shrink=0.45, aspect=14, pad=0.08)
cbar.set_label(r"Predicted Surface ($\hat{y}_{\text{tensor}}$)", labelpad=10)
cbar.ax.tick_params(labelsize=9)

# 5. Axis Labeling, Formatting & Perspectives
ax.set_title("Spatial Trend Surface (Tensor Spline)", fontweight="bold", pad=20)
ax.set_xlabel("Longitude (°)", labelpad=12)
ax.set_ylabel("Latitude (°)", labelpad=12)
# ax.set_zlabel("Predicted Value", labelpad=12)

# Explicitly set tick label sizes for X, Y, and Z axes uniformly
ax.tick_params(axis='both', which='major', labelsize=9)
ax.tick_params(axis='z', labelsize=9)

# Optimal viewing angle to clearly visualize gradients and trends
ax.view_init(elev=32, azim=-50)

# 6. Clean Background Panes and Professional Gridlines
ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

ax.xaxis._axinfo["grid"].update({"color": (0.8, 0.8, 0.8, 0.6), "linestyle": "--"})
ax.yaxis._axinfo["grid"].update({"color": (0.8, 0.8, 0.8, 0.6), "linestyle": "--"})
ax.zaxis._axinfo["grid"].update({"color": (0.8, 0.8, 0.8, 0.6), "linestyle": "--"})

# 7. Export Vector and Raster Formats
plt.savefig("tensor_mountain_surface.pdf", bbox_inches="tight", dpi=300)
plt.savefig("tensor_mountain_surface.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
from patsy import build_design_matrices, dmatrices
import patsy
from scipy.interpolate import griddata
from sklearn.linear_model import Ridge

# -------------------------------------------------------------------------
# 0. Academic Styling Configuration (Thesis Standards)
# -------------------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,  # Fixed typo here
    "ytick.major.width": 0.8,
})

# -------------------------------------------------------------------------
# 1. Prepare Panel Data & Fit the Defined Tensor Model
# -------------------------------------------------------------------------
gdf = sr.spatial_panel(mu=2, sigma=3,time=100,rho=0.7, seed=787)

gdf["time_idx"] = gdf["time"]

# Defined Tensor model
formula = "y_true ~ X_1 + X_2 + X_3 + te(cr(lat, df=6), cr(lon, df=6), constraints='center')"

# Generate design matrices
y, design = patsy.dmatrices(formula, gdf, return_type="dataframe")

# Fit the Ridge model
model = Ridge(alpha=0.01)
model.fit(design, y)

# Predict the overall spatial/trend surface
gdf['y_tensor'] = model.predict(design)

# -------------------------------------------------------------------------
# 2. Visualization 1: 2D Contour Slice (Thesis-Ready)
# -------------------------------------------------------------------------
target_time = 0

lon_grid = np.linspace(gdf["lon"].min(), gdf["lon"].max(), 200)
lat_grid = np.linspace(gdf["lat"].min(), gdf["lat"].max(), 200)
xx, yy = np.meshgrid(lon_grid, lat_grid)

grid_df = pd.DataFrame(
    {
        "lon": xx.ravel(),
        "lat": yy.ravel(),
        "X_1": 0,
        "X_2": 0,
        "X_3": 0,
    }
)

X_grid = build_design_matrices([design.design_info], grid_df, return_type="dataframe")[0]

spatial_tensor_cols = [col for col in design.columns if "te(" in col or "cr(" in col]
st_indices = [design.columns.get_loc(col) for col in spatial_tensor_cols]

st_weights = model.coef_.ravel()[st_indices]
spatial_surface = (
    X_grid[spatial_tensor_cols].values @ st_weights
).reshape(xx.shape)

# Set figure size optimized for standard thesis formatting (e.g., ~6 inches wide)
fig, ax = plt.subplots(figsize=(6.5, 5.5))

# Use a perceptually uniform colormap (e.g., 'viridis' or 'plasma')
contour = ax.contourf(
    xx, yy, spatial_surface, levels=30, cmap="viridis", alpha=0.9
)

# Add a properly proportioned colorbar using axes divider
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.15)
cbar = fig.colorbar(contour, cax=cax)
cbar.set_label(
    r"Spatial Tensor Contribution ($f(\text{lat}, \text{lon})$)",
    fontsize=10,
    labelpad=8,
)
cbar.ax.tick_params(labelsize=9)

slice_mask = gdf["time_idx"] == target_time
gdf_slice = gdf[slice_mask]

# Scatter plot of observed points overlaying the surface
scatter = ax.scatter(
    gdf_slice["lon"],
    gdf_slice["lat"],
    c=gdf_slice["y_true"],
    cmap="viridis",
    edgecolors="white",
    linewidths=0.6,
    s=45,
    alpha=0.95,
)

# Clean, professional labels and grid
ax.set_title(
    f"Spatial Tensor Product Effect (Time Index $t = {target_time}$)",
    pad=12,
    weight="bold"
)
ax.set_xlabel("Longitude", labelpad=8)
ax.set_ylabel("Latitude", labelpad=8)
ax.set_aspect("equal", adjustable="box")

# Fine-tune tick parameters and frame spines
ax.tick_params(direction="in", top=True, right=True)
for spine in ax.spines.values():
    spine.set_linewidth(0.8)

plt.tight_layout()

# Save vector format for thesis compilation (LaTeX/PDF workflow)
plt.savefig("spatial_tensor_effect.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import patsy
import statsmodels.api as sm
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# Assuming 'sr' is your custom module containing spatial_panel
# from your_module import sr

# Define range of sigma values to test
sigma_values = np.linspace(1, 50, 50)
true_params = np.array([5.0, 6.0, 7.0])  # True parameters for X_1, X_2, X_3

# Storage for results
metrics_data = []

for sig in sigma_values:
  # 1. Generate data with current sigma
  gdf = sr.spatial_panel(mu=2, sigma=sig, time=100, rho=0.7, seed=787)

  # --- Model A: Tensor Model ---
  formula = (
      "y_true ~ X_1 + X_2 + X_3 + te(cr(lat, df=6), cr(lon, df=6),"
      " constraints='center')"
  )
  y_tensor_true, design_tensor = patsy.dmatrices(formula, gdf)
  ridge_model = Ridge(alpha=0.01)
  ridge_model.fit(design_tensor, y_tensor_true)
  y_pred_tensor = ridge_model.predict(design_tensor).ravel()

  tensor_col_names = design_tensor.design_info.column_names
  t_indices = [
      tensor_col_names.index("X_1"),
      tensor_col_names.index("X_2"),
      tensor_col_names.index("X_3"),
  ]
  tensor_coefs = ridge_model.coef_.ravel()[t_indices]

  rmse_tensor = np.sqrt(mean_squared_error(gdf["y_true"], y_pred_tensor))
  r2_tensor = r2_score(gdf["y_true"], y_pred_tensor)
  diff_tensor = np.abs(tensor_coefs - true_params)

  metrics_data.append({
      "sigma": sig,
      "model": "Tensor",
      "RMSE": rmse_tensor,
      "R2": r2_tensor,
      "diff_X1": diff_tensor[0],
      "diff_X2": diff_tensor[1],
      "diff_X3": diff_tensor[2],
  })

  # --- Models B, C, D: Spatial Weight Models ---
  # Note: If your spatial weight models require specific spatial lag features,
  # make sure to include them in X_mat. Below assumes distinct handling per weight type if needed.
  for m in ["w_rook", "w_queen", "w_knn6"]:
    model_name = m.split("_")[1].upper()  # ROOK, QUEEN, KNN6

    X_mat = gdf[["X_1", "X_2", "X_3"]].values
    X_sm = sm.add_constant(X_mat)
    y_d = gdf["y_true"].values

    # Example placeholder: if ROOK/QUEEN/KNN utilize spatial weights differently, 
    # adjust the regression fit here. Otherwise, they yield identical OLS lines.
    ols_results = sm.OLS(y_d, X_sm).fit()
    y_pred_ols = ols_results.predict(X_sm)
    ols_coefs = ols_results.params[1:4]

    rmse_ols = np.sqrt(mean_squared_error(y_d, y_pred_ols))
    r2_ols = r2_score(y_d, y_pred_ols)
    diff_ols = np.abs(ols_coefs - true_params)

    metrics_data.append({
        "sigma": sig,
        "model": model_name,
        "RMSE": rmse_ols,
        "R2": r2_ols,
        "diff_X1": diff_ols[0],
        "diff_X2": diff_ols[1],
        "diff_X3": diff_ols[2],
    })

# Convert results to DataFrame
df_results = pd.DataFrame(metrics_data)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
models = df_results["model"].unique()

# 1. RMSE Plot
ax = axes[0, 0]
for mod in models:
  subset = df_results[df_results["model"] == mod]
  ax.plot(
      subset["sigma"], subset["RMSE"], marker="o", label=mod, linewidth=2
  )
ax.set_title("RMSE vs Sigma")
ax.set_xlabel("Sigma ($\sigma$)")
ax.set_ylabel("RMSE")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.6)

# 2. R^2 Plot
ax = axes[0, 1]
for mod in models:
  subset = df_results[df_results["model"] == mod]
  ax.plot(subset["sigma"], subset["R2"], marker="s", label=mod, linewidth=2)
ax.set_title("$R^2$ vs Sigma")
ax.set_xlabel("Sigma ($\sigma$)")
ax.set_ylabel("$R^2$")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.6)

# 3. Parameter Difference: X_1 (True = 5)
ax = axes[1, 0]
for mod in models:
  subset = df_results[df_results["model"] == mod]
  ax.plot(
      subset["sigma"], subset["diff_X1"], marker="^", label=mod, linewidth=2
  )
ax.set_title("Absolute Parameter Difference: $X_1$ (True = 5)")
ax.set_xlabel("Sigma ($\sigma$)")
ax.set_ylabel("|Estimated - True|")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.6)

# 4. Parameter Difference: X_2 (True = 6)
ax = axes[1, 1]
for mod in models:
  subset = df_results[df_results["model"] == mod]
  ax.plot(
      subset["sigma"], subset["diff_X2"], marker="d", label=mod, linewidth=2
  )
ax.set_title("Absolute Parameter Difference: $X_2$ (True = 6)")
ax.set_xlabel("Sigma ($\sigma$)")
ax.set_ylabel("|Estimated - True|")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.6)

# 5. Parameter Difference: X_3 (True = 7)
ax = axes[2, 0]
for mod in models:
  subset = df_results[df_results["model"] == mod]
  ax.plot(subset["sigma"], subset["diff_X3"], marker="v", label=mod, linewidth=2)
ax.set_title("Absolute Parameter Difference: $X_3$ (True = 7)")
ax.set_xlabel("Sigma ($\sigma$)")
ax.set_ylabel("|Estimated - True|")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.6)

# Hide the empty 6th subplot
axes[2, 1].axis("off")

plt.tight_layout()
plt.show()